In [1]:
import importlib

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} is not installed. Installing via !pip ...")
        try:
            get_ipython().system(f"pip install {package_name}")
            print(f"{package_name} installed successfully.")
        except Exception as e:
            print(f"Failed to install {package_name}. Error:\n{e}")

packages = ["pyDOE2", "diversipy", "pygmo", "optproblems", "pymoo","GPy"]

for pkg in packages:
    ensure_package(pkg)

from google.colab import drive
drive.mount('/content/drive')

pyDOE2 is already installed.
diversipy is already installed.
pygmo is not installed. Installing via !pip ...
  Using cached pygmo-v2.19.0.tar.gz (3.0 MB)
ERROR: pygmo from https://files.pythonhosted.org/packages/e2/12/090ba61479f60d5177a0048736d09dc028b2d65063ed44cb952df506336f/pygmo-v2.19.0.tar.gz does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
pygmo installed successfully.
optproblems is already installed.
pymoo is already installed.
GPy is already installed.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.insert(1, '/content/drive/MyDrive/PhD 2025-1 Offline data-driven MOP under uncertainty/ TGPR-MO 2023')
from desdeo_emo.EAs.RVEA import RVEA
from desdeo_problem.Problem import DataProblem
from desdeo_problem.testproblems.TestProblems import test_problem_builder
from pyDOE2 import lhs
import numpy as np
from framework.treedGP_framework import run_treed_GP as treedGP
import time
import scipy.io
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.graph_objects as go


# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
# Plot for paper
def plot_exp(F, title=None):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig, ax = plt.subplots(figsize=(5, 5))

        ax.scatter(non_pareto_F[:, 0], non_pareto_F[:, 1],
                   s=10, color="#87CEEB", alpha=0.7)
        ax.scatter(pareto_F[:, 0], pareto_F[:, 1],
                   s=10, color="#FF7F0E", marker='D')

        ax.set_xlim(-150, 600)
        ax.set_ylim(-100, 500)
        ax.set_xlabel("f1", fontsize=25)
        #ax.set_ylabel("f2", fontsize=25)
        ax.tick_params(direction='out', length=5, width=1, colors='black')
        ax.tick_params(labelsize=18)
        ax.legend(frameon=False, fontsize=10, loc='best')
        ax.axvline(x=0, color='gray', linestyle='--', linewidth=1)
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)

        if title:
            ax.set_title(title, fontsize=25)

        fig.tight_layout()
        plt.savefig(f"{title}.pdf", dpi=300, bbox_inches='tight')
        plt.show()



# Plot: 2 Objs and pareto front
def plot_obj_2d(F, xlim=(0, 1), ylim=(0, 1),):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig = go.Figure(
            data=go.Scatter(
                x=F[:, 0],
                y=F[:, 1],
                mode='markers',
                name='Objective Values',
                marker=dict(size=6, color='#87CEEB', opacity=0.7)
            )
        )
        fig.add_trace(go.Scatter(
            x=pareto_F[:, 0],
            y=pareto_F[:, 1],
            mode='markers',
            name='Pareto Front',
            marker=dict(size=7, color='#FF7F0E', opacity=0.9, symbol='diamond')
    ))
        fig.update_layout(
            xaxis_title='f1',
            yaxis_title='f2',
            width=600,
            height=600,
            xaxis=dict(range=list(xlim)),
            yaxis=dict(range=list(ylim))
        )
    fig.show()



# Metrics: PICP
def metrics_picp(y_true, y_lower, y_upper):
    y_true = np.asarray(y_true)
    y_lower = np.asarray(y_lower)
    y_upper = np.asarray(y_upper)

    inside = (y_true >= y_lower) & (y_true <= y_upper)
    picp_per_objective = np.mean(inside, axis=0)
    picp_mean = np.mean(picp_per_objective)

    return picp_per_objective, picp_mean

def mean_std(arr):
    return np.mean(arr), np.std(arr)

In [4]:
problem_name = 'DTLZ1'

nvars = 10
nobjs = 2
nsamples = 2000
x_names = [f'x{i}' for i in range(1,nvars+1)]
y_names = [f'f{i}' for i in range(1,nobjs+1)]
row_names = ['lower_bound','upper_bound']
prob = test_problem_builder(problem_name, nvars, nobjs)
x_data = lhs(nvars, nsamples)
y_data = prob.evaluate(x_data)[0]
x_low = np.ones(nvars)*0
x_high = np.ones(nvars)

In [6]:
problem, total_points_per_model, total_points_per_model_sequence = treedGP(x_data, y_data, x_low, x_high)

Building trees...
Building leaf node GPs...
Building finished...


In [7]:
a = 0
evolver_opt = RVEA(problem, use_surrogates=True, n_iterations=100, population_size=100)
while evolver_opt.continue_evolution():
    evolver_opt.iterate()
    a+=1
    print(a)

f_real = evolver_opt.population.objectives

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100


**HV Indicator**

In [10]:
# Min-Max normalization
if problem_name == 'dtlz1' or 'DTLZ1':
  obj_min = np.array([0,0])
  obj_max = np.array([800,800])

# Ref points
ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('Min-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
print(f"f_real  HV = {hv.do(f_real_normalization):.6f}")



Min-Max normalization -> Min:  [0 0]
Min-Max normalization -> Max:  [800 800]
HV Reference points:  [1.1 1.1]
f_real  HV = 1.033148


In [11]:
# Plot
plot_obj_2d(f_real, xlim=(-100, 600), ylim=(-100, 600))
plot_obj_2d(f_real_normalization, xlim=(-0.1, 1), ylim=(-0.1, 1))